# 策略工作台：粘贴代码，运行回测

按从上到下的顺序操作，每个代码框按 **Shift + Enter** 运行。每次改策略，只需重新运行第2步和第3步。

**第一次打开也可以直接点顶部 Run → Run All Cells，运行默认5DR。** 星号 `[*]` 表示正在计算；等待左侧出现数字后再往下运行。最后按 Ctrl + S 保存。

## 第1步：准备数据（每次打开后运行一次）
选中下面代码框，按 Shift + Enter。默认读取你电脑上的课程日频数据。回测区间和资金可在这里修改。

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path(r'.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import qbt

data = qbt.load(r'D:\Downloads\feathers')
market = qbt.build_market(data, log=None)
START, END = '2022-01-04', '2025-06-30'
CAPITAL = 10_000_000
print('数据已准备好。接下来运行第2步的策略代码。')

## 第2步：在下面的代码框粘贴你的策略
**双击下面代码框，选中原有代码后替换。函数名保留为 `my_strategy`。**

这里已经放好5DR示例：每日收盘后按最近五个交易日的复权收益，选择收益最低的100只股票，下一交易日开盘尝试成交。五日是回看窗口，不是固定持有五日。

粘贴完成后按 Shift + Enter；这一步只是定义策略，暂时不会产生回测结果。

In [ ]:
def my_strategy(ctx):
    # 先筛股票池：20日平均成交额至少5000万元，报价年龄至少60日
    universe = ctx.universe(
        min_amt=50_000_000, min_listed=60,
        not_st=True, exclude_suspended=True)

    # 最近五个交易日的复权收益；引擎会在下一日开盘尝试成交
    return_5d = ctx.ret(5, lag=0)

    # 选收益最低的100只，等权配置，目标总仓位98%
    selected = universe.bottom(return_5d, 100)
    return ctx.long(selected, gross=0.98)


## 第3步：运行回测并查看结果
选中下面代码框，按 Shift + Enter，等待计算完成。收益、费用和图表会直接出现在代码框下方。

**修改第2步后，必须先重新运行第2步，再运行这里。** 仅修改文字而不执行，Python仍会使用上一次定义的策略。

In [ ]:
result = qbt.run(
    data, my_strategy, start=START, end=END, capital=CAPITAL,
    market=market, name='策略工作台')

print(result)
metrics = result.metrics()
print(f'实际成交 {metrics["成交笔数"]:,} 笔，总费用 {metrics["总费用"]:,.2f} 元')
report = qbt.report(result, path=str(ROOT / 'reports' / 'workbench'), plotly=False)
print('报告已保存：', report.path)
display(report)


## 常见修改与提示
- 把 `100` 改成 `50`：选择50只股票。
- 把 `gross=0.98` 改成 `gross=0.5`：目标总仓位50%。
- 把 `bottom` 改成 `top`：选择五日收益最高的股票，变成五日动量。
- 默认撮合仍考虑涨跌停、停牌、T+1、整手、费用和容量；实际持仓可能偏离目标。
- ST和上市年龄来自课程数据可用信息的推断/代理；无真实公司行动事件时用复权因子再投资近似。
- 该5DR在2022年至2025年上半年默认配置下累计收益约-62.01%，代码实现不保证盈利。
- 若页面提示选择内核，选 Python 3；出现红色报错时，从第1步依次重新运行。